# Task 60: Manual Fairness Metrics

Compute disparate impact, demographic-parity differences, and equalized-odds TPR/FPR gaps directly from the held-out confusion counts by the dataset's binary sex field.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

paths=[Path('heart.csv'),Path('Task_60_Manual_Fairness_Metrics/heart.csv'),Path('Heart_Disease_Tasks/Task_60_Manual_Fairness_Metrics/heart.csv')]
data=pd.read_csv(next(p for p in paths if p.is_file())).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y,group=data[features].to_numpy(float),data.target.to_numpy(int),data.sex.to_numpy(int)

def stratified_split_indices(labels,fraction,seed):
    rng=np.random.default_rng(seed);a=[];b=[]
    for label in np.unique(labels):
        ids=np.flatnonzero(labels==label);rng.shuffle(ids);cut=round(fraction*len(ids));a.extend(ids[:cut]);b.extend(ids[cut:])
    rng.shuffle(a);rng.shuffle(b);return np.array(a),np.array(b)
train_idx,test_idx=stratified_split_indices(y,.8,60)
X_train,X_test,y_train,y_test=X[train_idx],X[test_idx],y[train_idx],y[test_idx];groups_test=group[test_idx]
mean,scale=X_train.mean(0),X_train.std(0);scale[scale==0]=1
X_train,X_test=(X_train-mean)/scale,(X_test-mean)/scale
class ManualLogistic:
    def fit(self,x,y):
        self.w=np.zeros(x.shape[1]);self.b=0.
        for _ in range(1400):
            p=1/(1+np.exp(-np.clip(x@self.w+self.b,-35,35)));e=p-y;self.w-=.06*(x.T@e/len(y)+.02*self.w);self.b-=.06*e.mean()
        return self
    def predict(self,x,threshold=.5):return (1/(1+np.exp(-np.clip(x@self.w+self.b,-35,35)))>=threshold).astype(int)
model=ManualLogistic().fit(X_train,y_train)
threshold=.5;pred=model.predict(X_test,threshold)

def group_table(actual,predicted,group_values):
    rows=[]
    for label in np.unique(group_values):
        mask=group_values==label;a=actual[mask];p=predicted[mask]
        tp=int(((a==1)&(p==1)).sum());fp=int(((a==0)&(p==1)).sum());tn=int(((a==0)&(p==0)).sum());fn=int(((a==1)&(p==0)).sum())
        rows.append({'group':int(label),'n':len(a),'selection rate':float(p.mean()),'TPR':tp/(tp+fn) if tp+fn else np.nan,'FPR':fp/(fp+tn) if fp+tn else np.nan,'TP':tp,'FP':fp,'TN':tn,'FN':fn})
    return pd.DataFrame(rows).set_index('group')

def disparity_ratio(rates):
    maximum=float(np.max(rates));minimum=float(np.min(rates))
    return minimum/maximum if maximum>0 else np.nan
def max_gap(values):
    valid=np.asarray(values,dtype=float);valid=valid[np.isfinite(valid)]
    return float(valid.max()-valid.min()) if len(valid)>=2 else np.nan

table=group_table(y_test,pred,groups_test)
print('Manual group outcome table:');display(table)
print('Disparate impact (min selection rate / max):',round(disparity_ratio(table['selection rate']),4))
print('Demographic parity selection-rate gap:',round(max_gap(table['selection rate']),4))
print('Equalized-odds component gaps - TPR:',round(max_gap(table.TPR),4),'FPR:',round(max_gap(table.FPR),4))

# Compute the metrics directly from group-specific confusion cells as a verification.
for row in table.itertuples():
    assert np.isclose(row.TP/(row.TP+row.FN),row.TPR) if row.TP+row.FN else np.isnan(row.TPR)
    assert np.isclose(row.FP/(row.FP+row.TN),row.FPR) if row.FP+row.TN else np.isnan(row.FPR)
print('Equalized-odds rates verified against each group confusion matrix.')
print('Only the dataset binary sex field is available here; values are sample estimates, not legal certification or a comprehensive fairness audit.')

Manual group outcome table:


,n,selection rate,TPR,FPR,TP,FP,TN,FN
group,,,,,,,,
0,18,0.833333,1.000000,0.00,15,0,3,0
1,43,0.465116,0.888889,0.16,16,4,21,2


Disparate impact (min selection rate / max): 0.5581
Demographic parity selection-rate gap: 0.3682
Equalized-odds component gaps - TPR: 0.1111 FPR: 0.16
Equalized-odds rates verified against each group confusion matrix.
Only the dataset binary sex field is available here; values are sample estimates, not legal certification or a comprehensive fairness audit.
